In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection
from scipy.ndimage import gaussian_filter1d
from matplotlib import gridspec
 
# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

work_path = "M:/1confiProj/"
os.chdir(work_path)
from scripts.modelClassGPU import ConfiModel
from scripts.experiments import Experiment, DataHandler
#from scripts.evaluators import ModelEvaluator
from scripts.run_optimizers import gen_init_params,define_constraints,run_bads

In [5]:
# step 1: simulate fake data with given parameters

# which model to simulate data from?
exp_name = 'sep_ecc_percep'
exp = Experiment(exp_name=exp_name, cluster=False)
dh = DataHandler(cluster=False)
m_id = 2
model = ConfiModel(m_id = m_id, exp_config = exp.exp_config)

In [6]:
param_bounds = dict(zip(model.estParamsNames, model.bounds))
param_bounds

{'sig_Vrh': (0.001, 5),
 'sig_Vrl': (0.1, 20),
 'sig_A': (0.1, 20),
 'a': (1e-08, 50),
 'b': (1e-08, 10),
 'w_vis_h': (0.01, 1),
 'w_vis_l': (0.01, 1),
 'sig_P': (1, 60),
 'p_com_h': (0.0001, 0.9999),
 'p_com_l': (0.0001, 0.9999),
 'rng': (1e-08, 0.8)}

In [ ]:
nruns = 1
# params is a dict: {param_name: value}
x0s = [gen_init_params(bounds = model.p_bounds, estParamsNames = model.estParamsNames, split_data = exp.exp_config['split_data_f']) for _ in range(nruns)]
trialConds = model.getCondRep(50)  # simulate 50 trials

for i, x0 in enumerate(x0s):
    dat = model.run_simulation(x0, trialConds=trialConds)
    #df_sim = dh.organize_dat(dat = dat,part_id=2)


m:\1confiProj\scripts\experiments.py:308: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(remove_outliers)


In [10]:
dat


array([[-10.       , -10.       ,   1.       ,   1.       , -13.547924 ,
         25.779087 ],
       [-10.       , -10.       ,   1.       ,   1.       ,  -4.3179936,
         25.789703 ],
       [-10.       , -10.       ,   1.       ,   1.       ,  -8.098195 ,
         25.811604 ],
       ...,
       [ 10.       ,  10.       ,   2.       ,   0.       ,  11.898307 ,
         67.10259  ],
       [ 10.       ,  10.       ,   2.       ,   0.       ,  15.25772  ,
        126.32598  ],
       [ 10.       ,  10.       ,   2.       ,   0.       ,   4.2114887,
         67.10452  ]], shape=(5000, 6), dtype=float32)

In [ ]:
# prepare bin edges for model fitting

In [ ]:
#step 2 fit the model to the simulated data and recover the parameters
#define objective function
objf = exp.get_objf(dat, model, exp.exp_config['NLL_method'])
x0s = [gen_init_params(model.p_bounds, model.estParamsNames, True, exp.exp_config['split_data_f']) for _ in range(nruns)]
if exp.exp_config['split_data_f'] ==1:
    cons=None
else:
    cons = define_constraints(model.estParamsNames, 'BADS', True)

# define the save file name
sub_path = os.path.join(exp.data_handler.fit_res_path, exp.exp_name, 'partial', 'model_' + str(m_id), 'subject_' + str(sub_id))
random_ending = ''.join(random.sample(string.ascii_letters,6))
save_file_name = os.path.join(sub_path, f"fitted_sub_{sub_id}_m{m_id}_{optimizer}_f{i_fold}_vrel{vrel}_{random_ending}.pkl")

# save the estimated parameters name
with open(os.path.join(exp.data_handler.fit_res_path, exp.exp_name, 'partial', 'model_' + str(m_id), 'est_param_names.txt'), 'w') as f:
    for item in model.estParamsNames:
        f.write("%s\n" % item)

# pr = cProfile.Profile()
# pr.enable()

print("\n -- Start Optimization: method BADS --")
lower_bounds = [b[0] for b in model.bounds]
upper_bounds = [b[1] for b in model.bounds]
plausible_lower_bounds= [b[0] for b in model.p_bounds] 
plausible_upper_bounds= [b[1] for b in model.p_bounds]

run_bads(m_id, sub_id, i_fold, save_file_name, objf, x0s,lower_bounds=lower_bounds, 
                            upper_bounds=upper_bounds, 
                            plausible_lower_bounds=plausible_lower_bounds, 
                            plausible_upper_bounds=plausible_upper_bounds,
                            non_box_cons=cons)